# Serving — Export Gold -> CSV para Power BI
**LogiLake | D'LOGIA** — Capa Serving del Medallion Architecture

Este notebook exporta las tablas Gold como archivos CSV limpios,
listos para importar en Power BI Desktop o Power BI Service.

Flujo:
1. Leer tablas Delta Gold desde `./data/gold/`
2. Exportar como CSV single-file con pandas (`UTF-8`, header incluido)
3. Archivos disponibles en `./powerbi/data/` para Power BI

Ver instrucciones de conexion en `powerbi/README.md`.

In [1]:
# ── SparkSession con Delta Lake 3.1.0 (PySpark 3.5.0) ────────────────────────
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_serving")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

26/04/04 02:21:27 WARN Utils: Your hostname, JuanCamiloDev resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/04 02:21:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/devcontainers/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/devcontainers/.ivy2/cache
The jars for the packages stored in: /home/devcontainers/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e05505e5-9826-4c7a-bbd5-fc911c4628c0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central


	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 230ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-e05505e5-9826-4c7a-bbd5-fc911c4628c0
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/7ms)


26/04/04 02:21:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.0 | Delta Lake activo


In [2]:
# ── Rutas locales ─────────────────────────────────────────────────────────────
import os
from pyspark.sql import functions as F

BASE_PATH   = "/mnt/c/logilake/data"
GOLD_PATH   = f"{BASE_PATH}/gold"
EXPORT_PATH = "/mnt/c/logilake/data/serving"

os.makedirs(EXPORT_PATH, exist_ok=True)
print(f"Gold path:   {GOLD_PATH}")
print(f"Export path: {EXPORT_PATH}")

GOLD_TABLES = [
    "kpi_global",
    "kpi_monthly",
    "kpi_category",
    "kpi_nps",
    "kpi_seller_state",
]

Gold path:   /mnt/c/logilake/data/gold
Export path: /mnt/c/logilake/data/serving


In [3]:
# ── Exportar Gold -> CSV con pandas ──────────────────────────────────────────
# Se usa toPandas() para generar un CSV single-file limpio.
# Ventaja sobre coalesce(1).write.csv(): produce un archivo directamente
# sin subdirectorio, listo para importar en Power BI sin renombrar.

def export_to_csv(table_name: str) -> None:
    df = spark.read.format("delta").load(f"{GOLD_PATH}/{table_name}")

    # Convertir timestamps a string ISO para compatibilidad con Power BI
    for field in df.schema.fields:
        if str(field.dataType) in ("TimestampType()", "DateType()"):
            df = df.withColumn(field.name, F.col(field.name).cast("string"))

    out_file = f"{EXPORT_PATH}/{table_name}.csv"
    pdf = df.toPandas()
    pdf.to_csv(out_file, index=False, encoding="utf-8")

    size_kb = os.path.getsize(out_file) / 1024
    print(f"OK  {table_name:<25s}  {len(pdf):>6} filas  {size_kb:6.1f} KB  ->  {out_file}")


print("Exportando tablas Gold -> CSV...\n")
for table in GOLD_TABLES:
    try:
        export_to_csv(table)
    except Exception as e:
        print(f"ERR {table}: {e}")

print("\nExportacion completada.")
print(f"Archivos listos en: {os.path.abspath(EXPORT_PATH)}")

Exportando tablas Gold -> CSV...



26/04/04 02:21:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


OK  kpi_global                      1 filas     0.3 KB  ->  /mnt/c/logilake/data/serving/kpi_global.csv


OK  kpi_monthly                    25 filas     1.5 KB  ->  /mnt/c/logilake/data/serving/kpi_monthly.csv


OK  kpi_category                   71 filas     3.4 KB  ->  /mnt/c/logilake/data/serving/kpi_category.csv


OK  kpi_nps                         5 filas     0.2 KB  ->  /mnt/c/logilake/data/serving/kpi_nps.csv


OK  kpi_seller_state               22 filas     0.9 KB  ->  /mnt/c/logilake/data/serving/kpi_seller_state.csv

Exportacion completada.
Archivos listos en: /mnt/c/logilake/data/serving


In [4]:
# ── Resumen de archivos exportados ───────────────────────────────────────────
print(f"{'Archivo':<30} {'Tamanio':>10} {'Filas':>8}")
print("-" * 52)
for table in GOLD_TABLES:
    csv_path = f"{EXPORT_PATH}/{table}.csv"
    if os.path.exists(csv_path):
        import pandas as pd
        row_count = len(pd.read_csv(csv_path))
        size_kb   = os.path.getsize(csv_path) / 1024
        print(f"{table + '.csv':<30} {size_kb:>9.1f}KB {row_count:>8}")
    else:
        print(f"{table + '.csv':<30} {'NO ENCONTRADO':>20}")

print("\nSiguiente paso: abrir Power BI Desktop")
print("  Inicio -> Obtener datos -> Carpeta ->" , os.path.abspath(EXPORT_PATH))

Archivo                           Tamanio    Filas
----------------------------------------------------
kpi_global.csv                       0.3KB        1
kpi_monthly.csv                      1.5KB       25
kpi_category.csv                     3.4KB       71
kpi_nps.csv                          0.2KB        5
kpi_seller_state.csv                 0.9KB       22

Siguiente paso: abrir Power BI Desktop
  Inicio -> Obtener datos -> Carpeta -> /mnt/c/logilake/data/serving


In [5]:
# ── Preview kpi_global.csv ────────────────────────────────────────────────────
import pandas as pd

df_preview = pd.read_csv(f"{EXPORT_PATH}/kpi_global.csv")
print("Schema de kpi_global.csv:")
print(df_preview.dtypes)
print("\nDatos:")
print(df_preview.to_string(index=False))

Schema de kpi_global.csv:
total_orders                     int64
total_delivered                  int64
total_canceled                   int64
otif_rate_pct                  float64
avg_delivery_days_actual       float64
avg_delivery_days_estimated    float64
avg_delay_days                 float64
total_revenue_brl              float64
avg_order_value_brl            float64
avg_review_score               float64
avg_freight_ratio_pct          float64
gold_computed_at                object
dtype: object

Datos:
 total_orders  total_delivered  total_canceled  otif_rate_pct  avg_delivery_days_actual  avg_delivery_days_estimated  avg_delay_days  total_revenue_brl  avg_order_value_brl  avg_review_score  avg_freight_ratio_pct           gold_computed_at
        99441            96478             625          93.23                      12.5                         24.4           -11.9        16008872.12               160.99              4.09                   30.8 2026-04-04 02:20:31.365233


In [ ]:
# Sincronizar dashboard/data/ con los CSVs exportados
import shutil, glob

DASHBOARD_DATA = "/mnt/c/logilake/dashboard/data"
os.makedirs(DASHBOARD_DATA, exist_ok=True)

print("
Sincronizando dashboard/data/ ...")
copied = 0
for csv_file in sorted(glob.glob(f"{EXPORT_PATH}/*.csv")):
    shutil.copy(csv_file, DASHBOARD_DATA)
    fname = os.path.basename(csv_file)
    print(f"  {fname} -> dashboard/data/")
    copied += 1

print(f"
{copied} archivos copiados. Dashboard listo para refrescar.")
